# 🥈 Silver Layer: Step-by-Step Data Cleaning & Enrichment

Welcome to the **Silver Layer** notebook. This notebook is designed to be **clear, intuitive, and easy to teach/explain** step-by-step.

### **What happens in this notebook?**
1. **Step 1: Load Raw Data** from `bronze.py`.
2. **Step 2: Fix Company Names** (Mapping 46 aliases $\rightarrow$ 21 canonical company names).
3. **Step 3: Standardize Categories** (Grouping 19 messy category tags $\rightarrow$ 7 clean sectors).
4. **Step 4: Clean Dates** (Convert mixed date strings to `dd-mm-yyyy`, extract Year, Quarter, Month).
5. **Step 5: Convert Revenues to USD ($M)** (Handle £, €, ¥, parse ranges like `$10M - $20M` by midpoint, convert to millions).
6. **Step 6: Clean Authors & Word Count** (Fill missing authors, sort chronologically).
7. **Step 7: Merge Company Metadata** (Calculate `company_age` and `company_size_category`).
8. **Step 8: Final Verification & Export** (Export clean dataset to `data/processed/tech_news_clean.csv`).

--- 
### **Step 1: Ingest Raw Data from Bronze Layer**
We import `get_raw_data()` from `bronze.py` to load both the articles CSV and the company metadata JSON.

In [10]:
import pandas as pd
import numpy as np
import re
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from medallion.bronze import get_raw_data

# 1. Load raw data
df_articles_raw, df_companies_raw = get_raw_data()
df_articles = df_articles_raw.copy()
df_companies = df_companies_raw.copy()

print(f"✅ Loaded {len(df_articles)} raw articles and {len(df_companies)} metadata companies.")
display(df_articles.head(3))
display(df_companies.head(3))

✅ Loaded 750 raw articles and 21 metadata companies.


,article_id,title,company_name,published_date,category,revenue,summary,url,author,word_count
0,ART0001,Scale AI Raises Series D Funding Round,Scale AI,21-Feb-20,Financial Technology,NaN,The company demonstrated significant advances ...,https://technews.example.com/articles/1,John Smith,2569.0
1,ART0002,DataRobot Announces Breakthrough in Large Lang...,DataRobot,02/23/2023,Software,"£244,094,488",Strategic acquisition strengthens competitive ...,https://technews.example.com/articles/2,Alex Johnson,1878.0
2,ART0003,Meta AI Raises Series D Funding Round,Meta AI,17-02-2022,Analytics,NaN,Leadership outlines vision for responsible AI ...,https://technews.example.com/articles/3,John Smith,1981.0


,company_name,founded_year,headquarters,employee_count,industry,is_public,stock_ticker
0,OpenAI,2006,"Austin, TX",12793,Data Analytics,False,OPEN
1,Anthropic,2006,"San Francisco, CA",43747,FinTech,False,NaN
2,Google DeepMind,2017,"Austin, TX",24274,Data Analytics,False,NaN


--- 
### **Step 2: Company Name Standardization & Verification**
**Why?** In the raw articles, company names are written in different ways (e.g. `AWS`, `Amazon Web Services (AWS)`, `Facebook AI Research`, `DeepMind`).
**How?** We use a simple replacement dictionary to map them all to the official canonical name in metadata.

In [11]:
# 1. Define clean replacement dictionary
company_name_mapping = {
    'AWS': 'Amazon Web Services',
    'Amazon Web Services (AWS)': 'Amazon Web Services',    
    'CloudFlare': 'Cloudflare',
    'Data Robot': 'DataRobot',
    'Databricks Inc.': 'Databricks',
    'DeepMind': 'Google DeepMind',
    'Google Deepmind': 'Google DeepMind',
    'Facebook AI Research': 'Meta AI',
    'Meta AI Research': 'Meta AI',
    'Azure': 'Microsoft',
    'Microsoft Azure': 'Microsoft',
    'Mongo DB': 'MongoDB',
    'Nvidia': 'NVIDIA',
    'NVIDIA Corporation': 'NVIDIA',
    'Open AI': 'OpenAI',
    'OpenAI Inc.': 'OpenAI',
    'Palantir Technologies': 'Palantir',
    'Snowflake Inc.': 'Snowflake',
    'The Boring Company / SpaceX': 'SpaceX',
    'Stripe Inc.': 'Stripe'
}

# 2. Apply mapping to create company_name_clean
df_articles['company_name_clean'] = df_articles['company_name'].replace(company_name_mapping)

# 3. Add validation flag: does this company exist in official metadata?
canonical_companies = set(df_companies['company_name'])
df_articles['has_company_metadata'] = df_articles['company_name_clean'].isin(canonical_companies)

matched_count = df_articles[df_articles['has_company_metadata']]['company_name_clean'].nunique()
print(f"✅ Verification: {matched_count} / {len(canonical_companies)} canonical metadata companies matched (100% covered!)")
display(df_articles[['company_name', 'company_name_clean', 'has_company_metadata']].drop_duplicates().head(8))

✅ Verification: 21 / 21 canonical metadata companies matched (100% covered!)


,company_name,company_name_clean,has_company_metadata
0,Scale AI,Scale AI,True
1,DataRobot,DataRobot,True
2,Meta AI,Meta AI,True
3,Confluent,Confluent,True
4,Amazon Web Services,Amazon Web Services,True
5,NVIDIA,NVIDIA,True
6,Tesla,Tesla,True
7,Microsoft,Microsoft,True


--- 
### **Step 3: Clean Categories into a Standard Taxonomy**
**Why?** Articles use 19 different raw category labels (e.g. `Artificial Intelligence`, `AI & ML`, `Machine Learning`).
**How?** We map them into 7 standardized business sector groups.

In [12]:
# 1. Clean string whitespace
str_columns = ['article_id', 'title', 'category', 'summary', 'url', 'author']
for col in str_columns:
    if col in df_articles.columns:
        df_articles[col] = df_articles[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan, '': np.nan})

# 2. Category mapping dictionary
category_mapping = {
    'AI & ML': 'AI_ML',
    'Artificial Intelligence': 'AI_ML',
    'Machine Learning': 'AI_ML',
    'AI/ML': 'AI_ML',
    'Cloud': 'Cloud_Computing',
    'Cloud Services': 'Cloud_Computing',
    'Cloud Computing': 'Cloud_Computing',
    'Cybersecurity': 'Cybersecurity',
    'InfoSec': 'Cybersecurity',
    'Security': 'Cybersecurity',
    'Data Analytics': 'Data_Analytics',
    'Analytics': 'Data_Analytics',
    'Big Data': 'Data_Analytics',
    'FinTech': 'FinTech',
    'Finance': 'FinTech',
    'Financial Technology': 'FinTech',
    'Enterprise Software': 'Enterprise_Software',
    'Software': 'Enterprise_Software',
    'SaaS': 'SaaS'
}

df_articles['category_clean'] = df_articles['category'].replace(category_mapping)

print("--- Standardized Category Breakdown ---")
display(df_articles['category_clean'].value_counts().to_frame(name='Number of Articles'))

--- Standardized Category Breakdown ---


,Number of Articles
category_clean,
AI_ML,161
Data_Analytics,131
FinTech,124
Cloud_Computing,119
Cybersecurity,104
Enterprise_Software,77
SaaS,34


--- 
### **Step 4: Clean Published Dates & Extract Time Dimensions**

#### **How Date Parsing Works (Easy Explanation for Teaching):**
```python
pd.to_datetime(
    df_articles['published_date'], 
    format='mixed',    # Handles multiple formats (e.g. '21-Feb-20', '02/23/2023', 'October 19, 2022')
    dayfirst=False,    # Treats ambiguous '02/03/2023' as US style (MM/DD/YYYY)
    utc=True,          # Normalizes to standard UTC timezone for clean sorting
    errors='coerce'    # Converts any invalid/corrupted dates into NaT (null) instead of crashing
)
```

From the parsed datetime, we extract: **Clean Date (`dd-mm-yyyy`)**, **Year**, **Quarter (1-4)**, and **Month (1-12)**.

In [13]:
# 1. Convert to unified datetime
df_articles['published_date_dt'] = pd.to_datetime(
    df_articles['published_date'], 
    format='mixed', 
    dayfirst=False, 
    utc=True, 
    errors='coerce'
)

# 2. Extract clean date string (dd-mm-yyyy) and integer time dimensions
df_articles['published_date_clean'] = df_articles['published_date_dt'].dt.strftime('%d-%m-%Y')
df_articles['published_year'] = df_articles['published_date_dt'].dt.year.astype('Int64')
df_articles['published_quarter'] = df_articles['published_date_dt'].dt.quarter.astype('Int64')
df_articles['published_month'] = df_articles['published_date_dt'].dt.month.astype('Int64')
df_articles['published_year_month'] = df_articles['published_date_dt'].dt.strftime('%Y-%m')

print("--- Date Transformation Samples (Raw vs Cleaned) ---")
display(df_articles[[
    'published_date', 'published_date_clean', 'published_year', 
    'published_quarter', 'published_month', 'published_year_month'
]].head(6))

--- Date Transformation Samples (Raw vs Cleaned) ---


,published_date,published_date_clean,published_year,published_quarter,published_month,published_year_month
0,21-Feb-20,21-02-2020,2020,1,2,2020-02
1,02/23/2023,23-02-2023,2023,1,2,2023-02
2,17-02-2022,17-02-2022,2022,1,2,2022-02
3,13-12-2023,13-12-2023,2023,4,12,2023-12
4,12/28/2021,28-12-2021,2021,4,12,2021-12
5,26-10-2021,26-10-2021,2021,4,10,2021-10


--- 
### **Step 5: Multi-Currency Revenue Parsing & Normalization ($M USD)**

### 🗺️ **The Value Transformation Journey (Step-by-Step Visual Example):**

Let's trace how different messy raw values travel through each line of code:

| Raw Input `revenue` | 1. `extract_currency` | 2. `parse_revenue` (Base Amount) | 3. `revenue_usd` (FX Applied) | 4. `revenue_usd_M` (Final $M USD) |
| :--- | :---: | :---: | :---: | :---: |
| `'$1.480 billion'` | `'USD'` | `$1,480,000,000$` | `$1,480,000,000$` $(\times 1.0)$ | **`1480`** |
| `'£244,094,488'` | `'GBP'` | $£244,094,488$ | `$310,000,000$` $(\times 1.27)$ | **`310`** |
| `'€1,254,545,455'` | `'EUR'` | $€1,254,545,455$ | `$1,380,000,000$` $(\times 1.10)$ | **`1380`** |
| `'¥360,000,000,000'` | `'JPY'` | $¥360,000,000,000$ | `$2,400,000,000$` $(\div 150)$ | **`2400`** |
| `'$10M - $20M'` *(Range)* | `'USD'` | `$15,000,000$` *(Midpoint: $\frac{10+20}{2}$)* | `$15,000,000$` $(\times 1.0)$ | **`15`** |
| `'Not disclosed'` | `<NA>` | `<NA>` | `<NA>` | **`<NA>`** |

In [14]:
# =============================================================================
# 1. DEFINE CONVERSION RATES (FX RATES TO USD)
# =============================================================================
# Business rules:
# - EUR: multiply by 1.1
# - GBP: multiply by 1.27
# - JPY: divide by 150
FX_RATES_TO_USD = {
    'USD': 1.0,
    'GBP': 1.27,    # Example: £100 -> $127 USD
    'EUR': 1.1,     # Example: €100 -> $110 USD
    'JPY': 1 / 150  # Example: ¥15,000 -> $100 USD (divide by 150)
}


# =============================================================================
# 2. FUNCTION 1: DETECT THE CURRENCY
# =============================================================================
def extract_currency(val):
    # Example Input: '£244,094,488'  -> Output: 'GBP'
    # Example Input: '$1.480B'        -> Output: 'USD'
    # Example Input: 'Not disclosed'  -> Output: NaN
    if pd.isna(val) or not isinstance(val, str):
        return np.nan
    
    val = val.strip()
    # Check if text means missing/undisclosed
    if val.lower() in ['not disclosed', 'unknown', 'n/a', 'none', '-', '', 'nan']:
        return np.nan
    
    # Detect currency symbols or 3-letter currency codes
    if '£' in val or 'GBP' in val.upper(): 
        return 'GBP'  # British Pound
    elif '€' in val or 'EUR' in val.upper(): 
        return 'EUR'  # Euro
    elif '¥' in val or 'JPY' in val.upper(): 
        return 'JPY'  # Japanese Yen
    elif '$' in val or 'USD' in val.upper(): 
        return 'USD'  # US Dollar
        
    return 'USD'  # Default fallback to USD


# =============================================================================
# 3. FUNCTION 2: PARSE A SINGLE NUMBER STRING INTO BASE NUMERIC FLOAT
# =============================================================================
def parse_single_revenue(s):
    # Example Input: '$1.480 billion' -> Output: 1,480,000,000.0
    # Example Input: '$980.0M'        -> Output:   980,000,000.0
    # Example Input: '£244,094,488'   -> Output:   244,094,488.0
    if not s or pd.isna(s):
        return np.nan
    
    s = str(s).strip().lower()
    if s in ['not disclosed', 'unknown', 'n/a', 'none', '-', '', 'nan']:
        return np.nan
    
    # Step A: Identify the multiplier (Billion = 10^9, Million = 10^6, Thousand = 10^3)
    multiplier = 1.0
    if 'billion' in s or re.search(r'(\d|\.)\s*b\b', s) or s.endswith('b'): 
        multiplier = 1e9  # 1,000,000,000
    elif 'million' in s or re.search(r'(\d|\.)\s*m\b', s) or s.endswith('m') or 'm usd' in s:
        multiplier = 1e6  # 1,000,000
    elif 'thousand' in s or re.search(r'(\d|\.)\s*k\b', s) or s.endswith('k'): 
        multiplier = 1e3  # 1,000
        
    # Step B: Extract just the numbers and decimal point using Regex (removes symbols like $, £, commas)
    num_match = re.search(r'[\d,]+(?:\.\d+)?', s)
    if num_match:
        try: 
            # Clean commas: '244,094,488' -> '244094488' -> 244094488.0
            raw_number = float(num_match.group(0).replace(',', ''))
            return raw_number * multiplier
        except ValueError: 
            return np.nan
            
    return np.nan


# =============================================================================
# 4. FUNCTION 3: HANDLE SINGLE VALUES OR RANGE MIDPOINTS
# =============================================================================
def parse_revenue(val):
    # Example Input: '$10M - $20M' -> splits into ['$10M', '$20M'] -> calculates (10M + 20M)/2 -> Output: 15,000,000.0
    # Example Input: '$980.0M'     -> single number               -> Output: 980,000,000.0
    if pd.isna(val) or not isinstance(val, str):
        return np.nan
        
    val = val.strip()
    if val.lower() in ['not disclosed', 'unknown', 'n/a', 'none', '-', '', 'nan']:
        return np.nan
    
    # Check if the string is a range (contains '-' or 'to')
    if ' - ' in val or ' to ' in val.lower():
        # Split range into two parts
        parts = re.split(r'\s+-\s+|\s+to\s+', val, flags=re.IGNORECASE)
        parsed_numbers = [parse_single_revenue(p) for p in parts]
        valid_numbers = [p for p in parsed_numbers if pd.notna(p)]
        
        # Take the midpoint (average of valid numbers)
        if valid_numbers:
            return sum(valid_numbers) / len(valid_numbers)
        return np.nan
        
    # Single number value
    return parse_single_revenue(val)


# =============================================================================
# 5. APPLY THE 4-STEP PIPELINE ON THE DATAFRAME
# =============================================================================

# Journey Step 1: Detect Currency ('USD', 'GBP', 'EUR', 'JPY')
df_articles['revenue_currency'] = df_articles['revenue'].apply(extract_currency)

# Journey Step 2: Parse raw string to base numeric amount (resolving range midpoints)
df_articles['revenue_clean'] = df_articles['revenue'].apply(parse_revenue)

# Journey Step 3: Convert to normalized USD using exchange rates
# Formula: revenue_usd = revenue_clean * FX_RATE
# Example: £244,094,488 * 1.27 = $310,000,000 USD
df_articles['revenue_usd'] = df_articles.apply(
    lambda row: row['revenue_clean'] * FX_RATES_TO_USD.get(row['revenue_currency'], 1.0) 
    if pd.notna(row['revenue_clean']) else np.nan, 
    axis=1
)

# Journey Step 4: Scale to Millions USD and cast to clean integer (Int64)
# Formula: revenue_usd_M = round(revenue_usd / 1,000,000)
# Example: $310,000,000 / 1,000,000 = 310 (Int64)
df_articles['revenue_usd_M'] = (df_articles['revenue_usd'] / 1e6).round(0).astype('Int64')


# =============================================================================
# 6. DISPLAY VERIFICATION TABLE SHOWING THE VALUE JOURNEY
# =============================================================================
print("--- Revenue Value Journey Sample (Raw -> Currency -> Clean -> USD -> USD Millions) ---")
display(df_articles[[
    'revenue',           # 1. Raw Input (e.g. '£244,094,488')
    'revenue_currency',  # 2. Currency Detected ('GBP')
    'revenue_clean',     # 3. Base Amount (244094488.0)
    'revenue_usd',       # 4. USD Amount ($310,000,000.0)
    'revenue_usd_M'      # 5. Output Millions (310)
]].dropna().head(8))

--- Revenue Value Journey Sample (Raw -> Currency -> Clean -> USD -> USD Millions) ---


,revenue,revenue_currency,revenue_clean,revenue_usd,revenue_usd_M
1,"£244,094,488",GBP,2.440945e+08,3.100000e+08,310
3,$980.0M,USD,9.800000e+08,9.800000e+08,980
5,$16.800B,USD,1.680000e+10,1.680000e+10,16800
6,75000.0M USD,USD,7.500000e+10,7.500000e+10,75000
8,$1.480 billion,USD,1.480000e+09,1.480000e+09,1480
9,"€1,254,545,455",EUR,1.254545e+09,1.380000e+09,1380
10,$320.0M,USD,3.200000e+08,3.200000e+08,320
12,$32.000B,USD,3.200000e+10,3.200000e+10,32000


--- 
### **Step 6: Author Cleaning & Chronological Sorting**
1. Replace missing authors with `'Unknown'`.
2. Cast `word_count` to nullable integer `Int64`.
3. Preserve the `original_index` and sort chronologically.

In [15]:
# 1. Fill missing authors and standardize word count
df_articles['author'] = df_articles['author'].fillna('Unknown')
df_articles['word_count'] = pd.to_numeric(df_articles['word_count'], errors='coerce').astype('Int64')
df_articles['original_index'] = df_articles.index.astype('Int64')

# 2. Sort chronologically by published_date_dt
df_articles_clean = df_articles.sort_values(
    by=['company_name_clean', 'category_clean', 'author', 'published_date_dt']
).drop(columns=['published_date_dt']).reset_index(drop=True)

print(f"✅ Cleaned Articles: {len(df_articles_clean)} rows")
display(df_articles_clean[['original_index', 'company_name_clean', 'category_clean', 'author', 'published_date_clean', 'word_count', 'revenue_usd_M']].head(5))

✅ Cleaned Articles: 750 rows


,original_index,company_name_clean,category_clean,author,published_date_clean,word_count,revenue_usd_M
0,660,Airbnb,AI_ML,John Smith,18-01-2024,1885,7878
1,117,Airbnb,AI_ML,Sam Wilson,27-02-2020,1068,2600
2,270,Airbnb,AI_ML,Unknown,31-10-2022,1910,<NA>
3,503,Airbnb,AI_ML,Unknown,27-11-2023,988,7800
4,724,Airbnb,AI_ML,Unknown,08-04-2024,1588,<NA>


--- 
### **Step 7: Merge Company Metadata & Calculate Features**

#### **Formulas Added:**
1. **`company_age`** = `published_year - founded_year`
2. **`company_size_category`**:
   - **Small**: Employee count $< 10,000$
   - **Medium**: Employee count between $10,000$ and $30,000$
   - **Large**: Employee count $> 30,000$

In [16]:
# 1. Standardize company metadata types
df_companies['founded_year'] = pd.to_numeric(df_companies['founded_year'], errors='coerce').astype('Int64')
df_companies['employee_count'] = pd.to_numeric(df_companies['employee_count'], errors='coerce').astype('Int64')
df_companies['is_public'] = df_companies['is_public'].astype('boolean')

# 2. Columns to select
article_cols = [
    'article_id', 'original_index', 'company_name_clean', 'has_company_metadata', 'title', 
    'category', 'category_clean', 'author', 'published_date_clean', 
    'published_year', 'published_quarter', 'published_month', 'published_year_month', 
    'word_count', 'revenue_usd_M', 'summary', 'url'
]
company_cols = ['industry', 'headquarters', 'founded_year', 'employee_count', 'is_public', 'stock_ticker']

# 3. Left join articles with metadata on company name
df_merged_clean = pd.merge(
    df_articles_clean[article_cols],
    df_companies[['company_name'] + company_cols],
    left_on='company_name_clean',
    right_on='company_name',
    how='left'
).drop(columns=['company_name'])

# 4. Formula 1: company_age = published_year - founded_year
df_merged_clean['company_age'] = (df_merged_clean['published_year'] - df_merged_clean['founded_year']).astype('Int64')

# 5. Formula 2: company_size_category based on employee count
def assign_size_category(emp):
    if pd.isna(emp): return 'Unknown'
    elif emp < 10000: return 'Small'     # < 10,000
    elif emp <= 30000: return 'Medium'   # 10,000 through 30,000
    else: return 'Large'                 # > 30,000

df_merged_clean['company_size_category'] = df_merged_clean['employee_count'].apply(assign_size_category)

print("--- Sample Enriched Data with Age & Size Category ---")
display(df_merged_clean[[
    'article_id', 'company_name_clean', 'published_year', 'founded_year', 
    'company_age', 'employee_count', 'company_size_category', 'industry', 'is_public'
]].head(6))

--- Sample Enriched Data with Age & Size Category ---


,article_id,company_name_clean,published_year,founded_year,company_age,employee_count,company_size_category,industry,is_public
0,ART0661,Airbnb,2024,1999,25,19967,Medium,Data Analytics,False
1,ART0118,Airbnb,2020,1999,21,19967,Medium,Data Analytics,False
2,ART0271,Airbnb,2022,1999,23,19967,Medium,Data Analytics,False
3,ART0504,Airbnb,2023,1999,24,19967,Medium,Data Analytics,False
4,ART0725,Airbnb,2024,1999,25,19967,Medium,Data Analytics,False
5,ART0398,Airbnb,2022,1999,23,19967,Medium,Data Analytics,False


--- 
### **Step 8: Export Clean Dataset & Verification**
We save the final clean dataset to `data/processed/tech_news_clean.csv`.

In [17]:
# Export clean dataset to data/processed/tech_news_clean.csv
output_csv_path = 'data/processed/tech_news_clean.csv'

try:
    df_merged_clean.to_csv(output_csv_path, index=False)
    print(f"✅ Successfully exported {len(df_merged_clean)} rows × {len(df_merged_clean.columns)} columns to '{output_csv_path}'!")
except PermissionError:
    print(f"⚠️ Note: '{output_csv_path}' is open in Excel. Close it to overwrite directly.")

# Schema and Data Types Summary Table
print("\n=== Final Merged DataFrame Schema & Data Types ===")
display(pd.DataFrame({
    'Data Type': df_merged_clean.dtypes,
    'Non-Null Count': df_merged_clean.notnull().sum(),
    'Sample Value': df_merged_clean.iloc[0]
}))

✅ Successfully exported 750 rows × 25 columns to 'data/processed/tech_news_clean.csv'!

=== Final Merged DataFrame Schema & Data Types ===


,Data Type,Non-Null Count,Sample Value
article_id,str,750,ART0661
original_index,Int64,750,660
company_name_clean,str,750,Airbnb
has_company_metadata,bool,750,True
title,str,750,Airbnb Achieves Profitability Milestone
category,str,750,Artificial Intelligence
category_clean,str,750,AI_ML
author,str,750,John Smith
published_date_clean,str,750,18-01-2024
published_year,Int64,750,2024
